In [0]:
import pandas as pd
from pyspark.sql import functions as F

CATALOG = "my_living_index"

CONFIG = {
    "sources": {
        "cpi_monthly": "https://storage.dosm.gov.my/cpi/cpi_2d.parquet",
        "cpi_annual": "https://storage.dosm.gov.my/cpi/cpi_2d_annual.parquet",
        "hh_income": "https://storage.dosm.gov.my/hies/hh_income.parquet",
        "hh_income_state": "https://storage.dosm.gov.my/hies/hh_income_state.parquet",
        "hies_state": "https://storage.dosm.gov.my/hies/hies_state.parquet",
        "hies_percentile": "https://storage.dosm.gov.my/hies/hies_malaysia_percentile.parquet",
        "lfs_quarterly": "https://storage.dosm.gov.my/labour/lfs_qtr.parquet",
    },
    "cpi_base_year": 2010,
    "b40_threshold": 4850,
    "m40_upper": 10971
}

def load_parquet(url: str, table_name: str):
    print(f"Fetching {url}")
    pdf = pd.read_parquet(url)

    if "date" in pdf.columns:
        pdf["date"] = pdf["date"].astype(str)

    sdf = spark.createDataFrame(pdf)

    sdf = sdf.withColumn("_source_url", F.lit(url)) \
        .withColumn("_ingested_at", F.current_timestamp()) \
        .withColumn("_source_row_count", F.lit(len(pdf)))

    return sdf

INGESTION_MAP = {
    "cpi_monthly": f"{CATALOG}.bronze.bronze_cpi_monthly",
    "cpi_annual": f"{CATALOG}.bronze.bronze_cpi_annual",
    "hh_income": f"{CATALOG}.bronze.bronze_hh_income_national",
    "hh_income_state": f"{CATALOG}.bronze.bronze_hh_income_state",
    "hies_state": f"{CATALOG}.bronze.bronze_hies_state",
    "hies_percentile": f"{CATALOG}.bronze.bronze_hies_percentile",
    "lfs_quarterly": f"{CATALOG}.bronze.bronze_lfs_quarterly",
}

ingestion_results = []

for source_key, table_name in INGESTION_MAP.items():
    url = CONFIG["sources"][source_key]
    print(f"\n[{source_key}] to {table_name}")

    try:
        sdf = load_parquet(url, source_key)
        row_count = sdf.count()

        sdf.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(table_name)

        print(f"Written {row_count:,} rows to {table_name}")
        ingestion_results.append({"sources": source_key, "table": table_name, "rows": row_count, "status": "SUCCESS"})

    except Exception as e:
        print(f"FAILED: {e}")
        ingestion_results.append({"sources": source_key, "table": table_name, "rows": 0, "status": f"FAILED: {str(e)}"})

print("\n" + "="*60)
print("BRONZE INGESTION SUMMARY")
print("="*60)
summary_df = spark.createDataFrame(ingestion_results)
display(summary_df)